# RL-Based Real-Time Electricity Pricing

This notebook reproduces the main results from the paper.
An RL agent (PPO) learns to set **hourly retail electricity prices** for a cluster of
N = 100 consumers, with the joint objective of:

- Maximising the service provider's profit margin over day-ahead wholesale prices
- Reducing the aggregate **peak-to-average ratio (PAR)**
- Keeping the mean consumer bill close to the flat-rate baseline

Three pricing schemes are compared:

| Scheme   | Description |
|----------|-------------|
| **Flat** | Fixed 0.25 EUR/kWh at all times (baseline) |
| **TOU**  | Two-block time-of-use schedule (peak / off-peak) |
| **RL**   | Agent-generated real-time prices (PPO, `stable-baselines3`) |

Two consumer clusters are considered:

- **Cluster A** — homogeneous (all consumers follow pattern A)
- **Cluster B** — heterogeneous mix of pattern A and pattern B consumers

---
**Repository layout**
```
rl-dynamic-pricing/
├── src/
│   ├── environment.py   # ResponsiveBuildingEnv (Gymnasium)
│   ├── evaluation.py    # Benchmarking utilities
│   └── plotting.py      # Figure helpers
├── data/
│   └── README.md        # How to obtain / format the input data
├── outputs/figures/     # Generated figures (git-ignored)
└── experiment.ipynb     # ← you are here
```

## 0 · Setup

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

from src import (
    ResponsiveBuildingEnv,
    build_tou_prices,
    evaluate_flat_pricing,
    evaluate_price_sequence,
    evaluate_rl_agent,
    plot_demand_samples,
    plot_price_distribution,
    plot_pricing_comparison,
)

SEED = 42
np.random.seed(SEED)

# Experiment constants
N_CONSUMERS = 100  # number of buildings in the cluster
PRICE_MIN, PRICE_MAX = 0.20, 0.30  # EUR/kWh retail price bounds
SURCHARGE_THRESHOLD = 230.0  # kW — grid peak power limit
SURCHARGE_FACTOR = 0.02  # EUR/kW² — quadratic penalty coefficient
TRAIN_TIMESTEPS = 500_000

## 1 · Load data

The code expects two inputs:

1. **`bsl_dem`** — hourly net grid demand for consumer type A (kW), `pd.Series` with `DatetimeIndex`.
2. **`bsl_dem2`** — hourly net grid demand for consumer type B (kW), same format.
3. **`hwt`** — hourly day-ahead wholesale price (EUR/kWh), same `DatetimeIndex`.

Replace the loading block below with your own data source.
See `data/README.md` for the expected schema.

In [ ]:
# ── Load consumer type-A demand ──────────────────────────────────────────────
# bsl_dem is the NET grid consumption (after PV and battery) resampled to 1-hour
# resolution. Replace this block with your own loading logic.
#
# Example (using the original simulation outputs):
#   from parser import OutputParser
#   bsl      = OutputParser('./data/normal.dat')
#   other    = OutputParser('./data/statopt.dat')
#   bsl_pgrid = bsl.df.SingleFamilyWithSolarAndBattery.resample('1h').sum()
#   bsl_dem   = (bsl_pgrid
#                + other.df.pv_SingleFamilyWithSolarAndBattery.resample('1h').sum()
#                - (bsl.df.battery_charge_energy_... * 2.77778e-7).resample('1h').sum()
#                + (bsl.df.battery_discharge_energy_... * 2.77778e-7).resample('1h').sum())

# ── Placeholder: load from CSV exports ───────────────────────────────────────
bsl_dem = pd.read_csv("data/demand_type_a.csv", index_col=0, parse_dates=True).squeeze()
bsl_dem2 = pd.read_csv(
    "data/demand_type_b.csv", index_col=0, parse_dates=True
).squeeze()

# ── Load wholesale prices ─────────────────────────────────────────────────────
# Source: e.g. ENTSO-E Transparency Platform (France, 2018)
# Expected columns: 'Datetime (Local)', 'Price (EUR/MWhe)'
w_prices_raw = pd.read_csv("data/wholesale_prices.csv")
hwt = (
    w_prices_raw.rename(columns={"Datetime (Local)": "dt", "Price (EUR/MWhe)": "price"})
    .assign(dt=lambda df: df.dt.map(datetime.datetime.fromisoformat))
    .set_index("dt")["price"]
    * 1e-3  # MWh → kWh
)

print(f"Type-A demand : {bsl_dem.index.min().date()} → {bsl_dem.index.max().date()}")
print(f"Type-B demand : {bsl_dem2.index.min().date()} → {bsl_dem2.index.max().date()}")
print(f"Wholesale prices: {hwt.index.min().date()} → {hwt.index.max().date()}")

## 2 · Define training and test periods

In [ ]:
TRAIN_PERIOD = "2018-07"  # month used to fit the KDE demand model and train RL
TRAIN_PERIOD_B = slice("2018-08-01", "2018-08-30")  # type-B consumer training window
TEST_DAY = "2018-08-01"  # single day used for evaluation

pgrid_train_A = bsl_dem[TRAIN_PERIOD]
pgrid_train_B = bsl_dem2[TRAIN_PERIOD_B]
w_prices_train = hwt[TRAIN_PERIOD].resample("1h").mean().ffill()

pgrid_test = bsl_dem[TEST_DAY]  # used only for its DatetimeIndex
w_prices_test = hwt[TEST_DAY].resample("1h").mean().ffill()

print(f"Training hours (A): {len(pgrid_train_A)}")
print(f"Training hours (B): {len(pgrid_train_B)}")
print(f"Test hours         : {len(pgrid_test)}")

## 3 · Build training environments

Two environments are created, differing only in their consumer flexibility parameters:

| Parameter  | `env_hi` (ε = 0.40) | `env_lo` (ε = 0.15) |
|------------|---------------------|---------------------|
| `p_lambda` | 0.40                | 0.15                |
| `p_epsilon`| 0.40                | 0.15                |
| `p_alpha`  | 0.25 (both)         | 0.25 (both)         |

`p_lambda` controls the fraction of demand that responds to a high price (shedding).
`p_epsilon` controls the fraction shifted forward in time by a low price.

In [ ]:
CONSUMER_PRICE_BOUNDS = (PRICE_MIN, PRICE_MAX)

COMMON_PARAMS = dict(
    p_alpha=0.25,
    demand_t_minus_1=pgrid_train_A.iloc[-1],
)

env_hi = ResponsiveBuildingEnv(
    pgrid_train_A,
    w_prices_train,
    CONSUMER_PRICE_BOUNDS,
    p_lambda=0.40,
    p_epsilon=0.40,
    **COMMON_PARAMS
)
env_lo = ResponsiveBuildingEnv(
    pgrid_train_A,
    w_prices_train,
    CONSUMER_PRICE_BOUNDS,
    p_lambda=0.15,
    p_epsilon=0.15,
    **COMMON_PARAMS
)
env_B = ResponsiveBuildingEnv(
    pgrid_train_B,
    w_prices_train,
    CONSUMER_PRICE_BOUNDS,
    p_lambda=0.40,
    p_epsilon=0.40,
    **COMMON_PARAMS
)

## 4 · Generate synthetic test demand trajectories

For each of the N consumers we sample one demand trajectory from the fitted KDE model.
These trajectories are held fixed during evaluation so that all pricing schemes are
compared on the same realisations.

In [ ]:
flat_test_prices = pd.Series(0.0, index=pgrid_test.index)  # neutral action → no DR

generated_A, generated_B = [], []
for _ in range(N_CONSUMERS):
    dem_A, _, _ = env_hi.evaluate_price_sequence(
        flat_test_prices, backlog_t_minus_1=0.0, demand_t_minus_1=pgrid_train_A.iloc[-1]
    )
    generated_A.append(dem_A)

    dem_B, _, _ = env_B.evaluate_price_sequence(
        flat_test_prices, backlog_t_minus_1=0.0, demand_t_minus_1=pgrid_train_A.iloc[-1]
    )
    generated_B.append(dem_B)

ax = plot_demand_samples(generated_A, label="Type A", color="C9")
plot_demand_samples(generated_B, label="Type B", color="C3", ax=ax)
plt.tight_layout()
plt.show()

## 5 · Assemble test clusters

Each consumer gets its own `ResponsiveBuildingEnv` configured with:
- The sampled demand trajectory (replayed directly via `use_pgrid_directly=True`)
- Normalisation statistics from the **training** period (not the test day)

In [ ]:
def build_cluster_envs(
    epsilons: list[float],
    demand_trajectories: list[pd.Series],
    source_pgrid: pd.Series,
) -> list[ResponsiveBuildingEnv]:
    """Wrap each sampled demand trajectory in its own test environment."""
    envs = []
    pgrid_stats = (source_pgrid.mean(), source_pgrid.std())
    wprice_stats = (w_prices_test.mean(), w_prices_test.std())

    for eps, traj in zip(epsilons, demand_trajectories):
        env = ResponsiveBuildingEnv(
            traj,
            w_prices_train,
            CONSUMER_PRICE_BOUNDS,
            p_lambda=eps,
            p_epsilon=eps,
            p_alpha=0.25,
            demand_t_minus_1=source_pgrid.iloc[-1],
            use_pgrid_directly=True,
            prev_pgrid_window_stats=pgrid_stats,
            prev_wprices_window_stats=wprice_stats,
        )
        envs.append(env)
    return envs


# Cluster A: all type-A consumers; half high-epsilon, half low-epsilon
eps_split = [0.40] * (N_CONSUMERS // 2) + [0.15] * (N_CONSUMERS // 2)
test_A_envs = build_cluster_envs(eps_split, generated_A, pgrid_train_A)

# Cluster B: type-B demand with the same epsilon split
test_B_envs = build_cluster_envs(eps_split, generated_B, pgrid_train_B)

print(f"Cluster A: {len(test_A_envs)} consumers")
print(f"Cluster B: {len(test_B_envs)} consumers")

## 6 · Train RL agents

Two PPO agents are trained — one per flexibility level.
Training uses the **July** data via the KDE sampler (stochastic demand).

> **Note:** 100 k timesteps takes ≈ 1–2 min on a modern laptop CPU.
> Increase `TRAIN_TIMESTEPS` for better convergence.

In [ ]:
LOAD_EXISTING_MODELS = False
if not LOAD_EXISTING_MODELS:
    model_hi = PPO(
        "MlpPolicy",
        env_hi,
        gamma=0.958,
        verbose=0,
        tensorboard_log="./outputs/tb_logs/model_hi/",
    )
    model_lo = PPO(
        "MlpPolicy",
        env_lo,
        gamma=0.958,
        verbose=0,
        tensorboard_log="./outputs/tb_logs/model_lo/",
    )

    print("Training model_hi (ε = 0.40) ...")
    model_hi.learn(total_timesteps=TRAIN_TIMESTEPS, progress_bar=True)

    print("Training model_lo (ε = 0.15) ...")
    model_lo.learn(total_timesteps=TRAIN_TIMESTEPS, progress_bar=True)

    # Optionally save trained models
    model_hi.save("outputs/model_hi")
    model_lo.save("outputs/model_lo")
    print("Models saved.")
else:
    print("Loading models")
    model_hi = PPO.load("outputs/model_hi.zip")
    model_lo = PPO.load("outputs/model_lo.zip")

## 7 · Evaluate all pricing schemes

For each cluster we compute PAR, consumer bills, and service-provider profit under:
1. Flat pricing (baseline)
2. TOU pricing
3. RL pricing (each consumer paired with the model matching its epsilon)

In [ ]:
tou_prices = build_tou_prices(pgrid_test.index)

# Assign matching model to each consumer (first half → model_hi, second half → model_lo)
models_A = [model_hi] * (N_CONSUMERS // 2) + [model_lo] * (N_CONSUMERS // 2)

EVAL_KWARGS = dict(
    w_prices=w_prices_test,
    surcharge_threshold=SURCHARGE_THRESHOLD,
    surcharge_factor=SURCHARGE_FACTOR,
)

# --- Cluster A ---
print("─── Cluster A ───")
metrics_A_flat = evaluate_flat_pricing(test_A_envs, **EVAL_KWARGS)
metrics_A_tou = evaluate_price_sequence(test_A_envs, tou_prices, **EVAL_KWARGS)
metrics_A_rl = evaluate_rl_agent(test_A_envs, models_A, **EVAL_KWARGS)

for m in [metrics_A_flat, metrics_A_tou, metrics_A_rl]:
    print(m.summary())

print()

# --- Cluster B ---
print("─── Cluster B ───")
metrics_B_flat = evaluate_flat_pricing(test_B_envs, **EVAL_KWARGS)
metrics_B_tou = evaluate_price_sequence(test_B_envs, tou_prices, **EVAL_KWARGS)
metrics_B_rl = evaluate_rl_agent(test_B_envs, models_A, **EVAL_KWARGS)  # same RL models

for m in [metrics_B_flat, metrics_B_tou, metrics_B_rl]:
    print(m.summary())

## 8 · Figures

In [ ]:
# Extract per-consumer RL prices for the fan plot
rl_prices_A = [t.prices for t in test_A_envs]
rl_prices_B = [t.prices for t in test_B_envs]

# TOU prices in EUR/kWh (for plotting — act_to_price converts the normalised actions)
tou_prices_eur = env_hi.act_to_price(tou_prices)
tou_prices_eur = pd.Series(tou_prices_eur, index=pgrid_test.index)

In [ ]:
fig_A = plot_pricing_comparison(
    tou_prices_eur=tou_prices_eur,
    rl_prices_list=rl_prices_A,
    flat_metrics=metrics_A_flat,
    tou_metrics=metrics_A_tou,
    rl_metrics=metrics_A_rl,
    surcharge_threshold=SURCHARGE_THRESHOLD,
    y_lim_load=(80, 250),
    save_path="outputs/figures/cluster_A_result.png",
)
plt.show()

In [ ]:
fig_B = plot_pricing_comparison(
    tou_prices_eur=tou_prices_eur,
    rl_prices_list=rl_prices_B,
    flat_metrics=metrics_B_flat,
    tou_metrics=metrics_B_tou,
    rl_metrics=metrics_B_rl,
    surcharge_threshold=SURCHARGE_THRESHOLD,
    y_lim_load=(80, 300),
    save_path="outputs/figures/cluster_B_result.png",
)
plt.show()

In [ ]:
# Price fan plots
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
plot_price_distribution(rl_prices_A, color="C2", ax=axes[0])
axes[0].set_title("Cluster A — RL prices")
plot_price_distribution(rl_prices_B, color="C3", ax=axes[1])
axes[1].set_title("Cluster B — RL prices")
plt.tight_layout()
plt.savefig("outputs/figures/price_distributions.png", dpi=300)
plt.show()